Imports

In [6]:
import pandas as pd

Read Raw Data

In [7]:
mbs_raw_data_df = pd.read_csv('../data/raw/mbs_dataset_raw_sample.csv', 
                              dtype={"tax_ref_no": str,"natlidno": str,"mobilephone": str, "pr_mbr_no": str}
                              )

Validation

In [8]:
# Ndabezinhle fix

from validation import validate_all

# validate_all(mbs_raw_data_df)

Read Validated Data

In [9]:
mbs_data_df = pd.read_csv('../data/primary/mbs_dataset_validated.csv', 
                            dtype={"tax_ref_no": str, "natlidno": str, "mobilephone": str, "pr_mbr_no": str}
                            )

mbs_data_df = mbs_raw_data_df # Remove once validation code is complete

Generate Statement for each Member

In [10]:
from pathlib import Path

from PIL import Image
from statement_templates.saccawu import generate

old_mutual_logo = Image.open("../assets/old_mutual_header.png")
saccawu_logo = Image.open("../assets/saccawu_logo.png")

reporting_dt = mbs_data_df[mbs_data_df["acc_credit"].notna()]["pyrl_dt"].max()
reporting_dt = pd.to_datetime(reporting_dt)

start_dt = reporting_dt.replace(day=1) - pd.DateOffset(months=11)

# Loop through members to process each members statement
for case_mbr_key in mbs_data_df["case_mbr_key"].dropna().unique():
    
    # Filter mbs data between a period and current member
    mbs_data_df["pyrl_dt"] = pd.to_datetime(mbs_data_df["pyrl_dt"])
    mbs_data = mbs_data_df[mbs_data_df["pyrl_dt"].between(start_dt, reporting_dt) & (mbs_data_df["case_mbr_key"] == case_mbr_key)]
        
    output_path = Path(f"../statements/{case_mbr_key}_member_statement.pdf")

    generate.generate_statement(mbs_data, output_path, old_mutual_logo, saccawu_logo, reporting_dt, start_dt)